In [74]:
!pip install torch
!pip install torch_geometric

### Load Dataset

In [75]:
import pandas as pd

df = pd.read_csv("data/elite_data/enriched.csv")
df = df.sample(100000,random_state=42)

print(df.shape)

(100000, 30)


### Create Node ID Mappings

In [76]:
accounts = pd.Index(pd.concat([df["sender_account"], df["receiver_account"]]).unique())
devices = pd.Index(df["device_hash"].unique())
ips = pd.Index(df["ip_address"].unique())
locations = pd.Index(df["location"].unique())
merchants = pd.Index(df["merchant_category"].unique())

account_map = {v: i for i, v in enumerate(accounts)}
device_map = {v: i for i, v in enumerate(devices)}
ip_map = {v: i for i, v in enumerate(ips)}
location_map = {v: i for i, v in enumerate(locations)}
merchant_map = {v: i for i, v in enumerate(merchants)}

In [77]:
print("Accounts:", len(accounts))
print("Devices:", len(devices))
print("IPs:", len(ips))
print("Locations:", len(locations))
print("Merchants:", len(merchants))

Accounts: 179590
Devices: 99473
IPs: 99999
Locations: 8
Merchants: 8


### Build Transaction Edge Index

#### Transaction edges (Account → Account)

In [78]:
import torch
import numpy as np
src = df["sender_account"].map(account_map).values
dst = df["receiver_account"].map(account_map).values

transaction_edge_index = torch.tensor(np.vstack((src, dst)), dtype=torch.long)

fraud_labels = torch.tensor(df["is_fraud"].astype(int).values)

#### Device edges (Account → Device)

In [79]:
src = df["sender_account"].map(account_map).values
dst = df["device_hash"].map(device_map).values

device_edge_index = torch.tensor(np.vstack((src, dst)), dtype=torch.long)

#### IP edges (Account → IP)

In [80]:
src = df["sender_account"].map(account_map).values
dst = df["ip_address"].map(ip_map).values

ip_edge_index = torch.tensor(np.vstack((src, dst)), dtype=torch.long)

#### Location edges (Account → Location)

In [81]:
src = df["sender_account"].map(account_map).values
dst = df["location"].map(location_map).values

location_edge_index = torch.tensor(np.vstack((src, dst)), dtype=torch.long)

#### Merchant edges (Account → Merchant)

In [82]:
src = df["sender_account"].map(account_map).values
dst = df["merchant_category"].map(merchant_map).values

merchant_edge_index = torch.tensor(np.vstack((src, dst)), dtype=torch.long)

### Account Node Features

In [83]:
from sklearn.preprocessing import StandardScaler
account_features = df.groupby("sender_account")[[
    "velocity_score",
    "spending_deviation_score",
    "geo_anomaly_score",
    "tx_count_24h",
    "amt_sum_24h",
    "device_fraud_rate",
    "ip_fraud_rate"
]].mean()

device_features = df.groupby("device_hash")[["device_fraud_rate"]].mean()
device_features=device_features.reindex(devices).fillna(0)
device_x=torch.tensor(device_features.values,dtype=torch.float)

ip_features = df.groupby("ip_address")[["ip_fraud_rate"]].mean()
ip_features=ip_features.reindex(ips).fillna(0)
ip_x=torch.tensor(ip_features.values,dtype=torch.float)

scaler=StandardScaler()
account_features=pd.DataFrame(scaler.fit_transform(account_features),index=account_features.index,columns=account_features.columns)
account_features = account_features.reindex(accounts).fillna(0)

account_x = torch.tensor(account_features.values, dtype=torch.float)

### Build HeteroData Graph

In [84]:
from torch_geometric.data import HeteroData

data = HeteroData()

# node features
data["account"].x = account_x
data["account"].num_nodes = len(accounts)

data["device"].x = device_x
data["device"].num_nodes=len(devices)
data["ip"].x=ip_x
data["ip"].num_nodes=len(ips)
data["location"].num_nodes = len(locations)
data["merchant"].num_nodes = len(merchants)

# edges
data["account","transaction","account"].edge_index = transaction_edge_index
data["account","transaction","account"].edge_label = fraud_labels

# ADD THIS
data["account","transaction","account"].edge_attr = torch.tensor(
    df["amount_wins"].values,
    dtype=torch.float
).view(-1,1)

data["account","uses_device","device"].edge_index = device_edge_index
data["account","uses_ip","ip"].edge_index = ip_edge_index
data["account","located_at","location"].edge_index = location_edge_index
data["account","purchases","merchant"].edge_index = merchant_edge_index

### Save Graph

In [85]:
torch.save(data, "data/fraud_graph.pt")

In [86]:
print(data)

HeteroData(
  account={
    x=[179590, 7],
    num_nodes=179590,
  },
  device={
    x=[99473, 1],
    num_nodes=99473,
  },
  ip={
    x=[99999, 1],
    num_nodes=99999,
  },
  location={ num_nodes=8 },
  merchant={ num_nodes=8 },
  (account, transaction, account)={
    edge_index=[2, 100000],
    edge_label=[100000],
    edge_attr=[100000, 1],
  },
  (account, uses_device, device)={ edge_index=[2, 100000] },
  (account, uses_ip, ip)={ edge_index=[2, 100000] },
  (account, located_at, location)={ edge_index=[2, 100000] },
  (account, purchases, merchant)={ edge_index=[2, 100000] }
)
